# Improvements Learning Notebook

Proposed-extension concepts; each code cell runs the minimal witness of the idea.

# Control-Variate Drift Correction
**Level:** `extension`
**Concept ID:** `01-control-variate-drift-correction`
**Graph:** `improvements`
**Prerequisites:** [paper:08-heterogeneity-gap-covariance](../../paper/concepts/08-heterogeneity-gap-covariance.md), [control variates](https://github.com/pleyva2004/math-foundations/blob/main/concepts/control-variates.md)
**Used by:** downstream nodes

## Plain-English intro
FedAvg lets each client take several local gradient steps before averaging, but on non-IID data each client drifts toward *its own* optimum instead of the global one. A control variate is a per-client correction vector that, added to every local step, re-centers that client's trajectory onto the global gradient direction. This is the SCAFFOLD fix, and it directly cancels the heterogeneity drift identified in paper node 08.

## Symbols you'll see
| Symbol | Read aloud as | Meaning |
|--------|---------------|---------|
| $w$ | "w" | Model parameter vector being updated locally <!-- TODO add to foundations --> |
| $\eta$ | "eta" | Local learning rate <!-- TODO add to foundations --> |
| $g_{\mathrm{local}}(w)$ | "g-local of w" | Client's local gradient at the current local iterate $w$ <!-- TODO add to foundations --> |
| $c_k$ | "c-sub-k" | Client $k$'s control variate (its drift estimate) <!-- TODO add to foundations --> |
| $c$ | "c" | Server control variate, average of selected $c_k$ <!-- TODO add to foundations --> |
| $S_t$ | "S-sub-t" | Set of clients selected in round $t$ <!-- TODO add to foundations --> |
| $H_k=\nabla^2 F_k(w_t)$ | "H-sub-k" | Client $k$'s local Hessian at $w_t$ <!-- TODO add to foundations --> |
| $g_k=\nabla F_k(w_t)$ | "g-sub-k" | Client $k$'s local gradient at the shared point <!-- TODO add to foundations --> |

## Formal definition
$$
\text{Local step (client } k):\quad w \leftarrow w-\eta\bigl(g_{\mathrm{local}}(w)-c_k+c\bigr),
\qquad c=\frac{1}{|S_t|}\sum_{k\in S_t}c_k .
$$
In expectation the $-c_k+c$ term replaces the locally-biased direction with the global one, cancelling the leading client-heterogeneity drift
$$
\Delta=\eta^2\,\mathrm{Cov}_k\!\bigl(H_k,g_k\bigr)+O(\eta^3),
$$
so the multi-step round recovers $w\leftarrow w-\eta\,\nabla f(w)$ to leading order.

## Why this matters
Operationalizes the heterogeneity-gap finding (paper node 08): it cancels the leading $\eta^2\mathrm{Cov}_k(H_k,g_k)$ drift of Eq. (3.5) in `02-math-deep-dive.md`, and is the highest-leverage proposal M.1 of `05-improvements.tex`.

## Code
The aligned runnable demo lives at [`../code/01-control-variate-drift-correction.py`](../code/01-control-variate-drift-correction.py). It demonstrates the concept on a finite/low-dimensional example, runnable in <30s on CPU.

**Expected output preview:**
```
Global gradient grad f(w_t) at shared point w_t=[0.7 0.7]:
  g_global = [-0.275 -0.275]
Per-client effective local direction vs global gradient:
```

## Try it yourself
- Exercise 1: Set the two Hessians equal (`H[1] = H[0]`) — the homogeneous limit. Confirm the plain and corrected deviations both collapse, since $\mathrm{Cov}_k(H_k,g_k)=0$.
- Exercise 2: Increase `tau` (local steps) and `eta`. Watch the *plain* drift grow with $\eta^2\tau$ while the corrected direction stays anchored to the global gradient.

## Further reading
- Karimireddy et al., "SCAFFOLD: Stochastic Controlled Averaging for Federated Learning," ICML 2020.


In [ ]:
"""Control-Variate Drift Correction -- concept 01-control-variate-drift-correction
of the improvements learning map.

On a 2-client heterogeneous quadratic, SCAFFOLD-style control variates re-center
each client's multi-step local trajectory onto the GLOBAL gradient direction,
cancelling the leading client-heterogeneity drift eta^2*Cov_k(H_k, g_k) (eq. 3.5
of 02-math-deep-dive.md; M.1 of 05-improvements.tex).

Runnable code analog of concepts/01-control-variate-drift-correction.py
(see concepts/01-control-variate-drift-correction.md).
"""
import numpy as np


def main():
    np.random.seed(0)

    # Two heterogeneous local quadratics  F_k(w) = 0.5 (w-b_k)^T H_k (w-b_k).
    # grad F_k(w) = H_k (w - b_k);  Hessians H_k differ => heterogeneity.
    H = [np.array([[3.0, 0.0], [0.0, 0.5]]),
         np.array([[0.5, 0.0], [0.0, 3.0]])]
    b = [np.array([1.0, 0.0]), np.array([0.0, 1.0])]
    nk = np.array([1.0, 1.0])          # equal client sizes -> weights n_k/n = 1/2
    alpha = nk / nk.sum()

    def gk(k, w):                      # local gradient g_local(w) for client k
        return H[k] @ (w - b[k])

    w0 = np.array([0.7, 0.7])          # shared broadcast point w_t
    eta, tau = 0.10, 5                 # local lr and number of local steps

    # GLOBAL gradient at the shared point: grad f(w0) = sum_k alpha_k g_k(w0)  (eq. 2.2)
    g_global = sum(alpha[k] * gk(k, w0) for k in range(2))

    # Control variates: c_k = g_k(w0), server c = mean_k c_k  (eq. M.1).
    c = [gk(k, w0) for k in range(2)]
    c_server = sum(alpha[k] * c[k] for k in range(2))

    def local_traj(k, corrected):
        w = w0.copy()
        for _ in range(tau):
            d = gk(k, w)
            if corrected:
                d = d - c[k] + c_server     # heterogeneity-corrected direction
            w = w - eta * d
        return (w0 - w) / (tau * eta)        # avg per-step direction client moved

    # Compare each client's effective local direction to the global gradient.
    print("Global gradient grad f(w_t) at shared point w_t={}:".format(w0))
    print("  g_global = {}".format(np.round(g_global, 4)))
    print("Per-client effective local direction vs global gradient:")
    dev_plain, dev_corr = 0.0, 0.0
    for k in range(2):
        dp = local_traj(k, corrected=False)
        dc = local_traj(k, corrected=True)
        ep = np.linalg.norm(dp - g_global)
        ec = np.linalg.norm(dc - g_global)
        dev_plain += alpha[k] * ep
        dev_corr += alpha[k] * ec
        print("  client {}: plain dir ={}  dev={:.4f} | corrected dir ={}  dev={:.4f}"
              .format(k, np.round(dp, 3), ep, np.round(dc, 3), ec))

    print("Client-weighted deviation from global gradient:")
    print("  plain local steps      : {:.4f}".format(dev_plain))
    print("  control-variate steps  : {:.4f}".format(dev_corr))
    assert dev_corr < dev_plain, "corrected direction must track global gradient better"
    print("VERDICT: PASS -- control variates make local steps track grad f "
          "({:.2f}x smaller drift).".format(dev_plain / dev_corr))


if __name__ == "__main__":
    main()


# --- run the witness ---
main()


# Unbiased Aggregation Estimators
**Level:** `extension`  *(novice | intermediate | advanced | graduate | frontier | paper | extension)*
**Concept ID:** `02-unbiased-aggregation-estimators`
**Graph:** `improvements`
**Prerequisites:** [paper:10-aggregation-erratum](../../paper/concepts/10-aggregation-erratum.md), [Horvitz-Thompson estimator](https://github.com/pleyva2004/math-foundations/blob/main/concepts/horvitz-thompson-estimator.md)
**Used by:** downstream nodes

## Plain-English intro
FedAvg's corrected server step averages the selected clients' models weighted by their data counts and normalized by the total examples *in the drawn set*. Because that normalizer is itself random and tends to be large exactly when a big client is sampled, the average systematically over-weights large clients — it is a Hajek *ratio* estimator, biased under client imbalance. Swapping the random normalizer for the fixed inclusion probability (Horvitz-Thompson) restores exact unbiasedness for any imbalance.

## Symbols you'll see
| Symbol | Read aloud as | Meaning |
|--------|---------------|---------|
| $v_k$ | "v-sub-k" | Client $k$'s returned model $w^k_{t+1}$ <!-- TODO add to foundations --> |
| $n_k$ | "n-sub-k" | Number of examples on client $k$ <!-- TODO add to foundations --> |
| $n=\sum_k n_k$ | "n" | Total examples over all $K$ clients <!-- TODO add to foundations --> |
| $S$ | "S" | Random set of selected clients, $\lvert S\rvert=m$ <!-- TODO add to foundations --> |
| $m_t=\sum_{k\in S} n_k$ | "m-sub-t" | Random normalizer: examples on the selected clients <!-- TODO add to foundations --> |
| $m,\,K$ | "m, K" | Clients sampled per round; total clients <!-- TODO add to foundations --> |
| $p_k=m/K$ | "p-sub-k" | Uniform inclusion probability of client $k$ <!-- TODO add to foundations --> |
| $\bar v$ | "v-bar" | Target: full size-weighted mean $\sum_k\frac{n_k}{n}v_k$ <!-- TODO add to foundations --> |

## Formal definition
$$
\bar v=\sum_{k=1}^{K}\tfrac{n_k}{n}\,v_k,\qquad
\underbrace{\sum_{k\in S}\tfrac{n_k}{m_t}\,v_k}_{\text{self-normalized (Hajek ratio, }O(1/m)\text{ bias)}}\!,\qquad
\underbrace{g_{\mathrm{HT}}(S)=\tfrac{K}{m}\sum_{k\in S}\tfrac{n_k}{n}\,v_k}_{\mathbb{E}_S[g_{\mathrm{HT}}]=\bar v\ \text{(exactly unbiased)}}.
$$
With $p_k=m/K$, inverse-probability weighting gives $\mathbb{E}_S[g_{\mathrm{HT}}]=\sum_k p_k\frac{1}{p_k}\frac{n_k}{n}v_k=\bar v$. Size-proportional sampling ($p_k\propto n_k$) makes the plain mean over $S$ unbiased too.

## Why this matters
Fixes the $O(1/m)$ bias of the erratum-corrected average (paper node 10) under client imbalance; appears as M.2, Eqs. (a)–(b) of 05-improvements.tex and the §5–§6 ratio-vs-IPW distinction in 02-math-deep-dive.md.

## Code
The aligned runnable demo lives at [`../code/02-unbiased-aggregation-estimators.py`](../code/02-unbiased-aggregation-estimators.py). It demonstrates the concept on a finite/low-dimensional example, runnable in <30s on CPU.

**Expected output preview:**
```
Unbiased aggregation under client imbalance (FedAvg, 05-improvements.tex M.2)
K=100 clients, m=C*K=10, dim=8, 200,000 Monte-Carlo draws
Target vbar = sum_k (n_k/n) v_k; we report bias ||E[agg]-vbar||
```

## Try it yourself
- Exercise 1: Reduce the size spread in `make_population` toward balanced ($n_k\equiv$ const) and watch the self-normalized bias collapse to the Monte-Carlo floor.
- Exercise 2: Add the size-proportional sampling fix ($p_k\propto n_k$, then plain mean) and confirm it is unbiased too.

## Further reading
- McMahan et al. 2017, *Communication-Efficient Learning of Deep Networks*, arXiv:1602.05629 (Algorithm 1, footnotes 2 & 4).
- 05-improvements.tex, section M.2; 02-math-deep-dive.md, §5–§6.


In [ ]:
"""Unbiased Aggregation Estimators -- concept 02-unbiased-aggregation-estimators
of the improvements learning map.

FedAvg's erratum-corrected average sum_{k in S} (n_k/m_t) v_k is a Hajek ratio
estimator (biased under client imbalance); the Horvitz-Thompson form
(K/m) sum_{k in S} (n_k/n) v_k is exactly unbiased. This Monte-Carlo witness
prints each estimator's bias norm: self-normalized >> 0 while HT ~ 0.
Runnable code analog of concepts/02-unbiased-aggregation-estimators.md.
"""
import numpy as np

np.random.seed(0)
K = 100              # number of clients
C = 0.1              # client fraction -> m = C*K clients sampled per round
M = max(1, int(round(C * K)))   # = 10
D = 8                # dimension of each client's returned vector v_k
N_DRAWS = 200_000    # Monte-Carlo uniform size-M subsets


def make_population(imbalanced):
    """Return (sizes n_k of shape (K,), vectors v_k of shape (K, D)).
    v_k is correlated with n_k so size-induced bias is visible (worst case)."""
    if imbalanced:
        sizes = np.round(np.geomspace(10.0, 1000.0, K)).astype(np.int64)
    else:
        sizes = np.full(K, 100, dtype=np.int64)
    s = (sizes - sizes.min()) / (sizes.max() - sizes.min() + 1e-12)   # in [0,1]
    base = np.random.normal(0.0, 1.0, size=(K, D))
    axis = np.random.normal(0.0, 1.0, size=D)
    v = base + 3.0 * np.outer(s, axis)        # large clients shifted along axis
    return sizes, v


def true_target(sizes, v):
    """vbar = sum_k (n_k/n) v_k -- the full size-weighted mean (the estimand)."""
    return (sizes / sizes.sum()) @ v


def mc_uniform(sizes, v):
    """E[self-normalized] and E[Horvitz-Thompson] over uniform size-M subsets."""
    n, p = sizes.sum(), M / K
    acc_sn = np.zeros(D)
    acc_ht = np.zeros(D)
    done = 0
    while done < N_DRAWS:
        b = min(20_000, N_DRAWS - done)
        keys = np.random.random((b, K))
        S = np.argpartition(keys, M, axis=1)[:, :M]        # (b, M) uniform subsets
        nk, vk = sizes[S], v[S]                            # (b, M), (b, M, D)
        mt = nk.sum(axis=1, keepdims=True)                 # RANDOM normalizer
        acc_sn += np.einsum("bm,bmd->bd", nk / mt, vk).sum(axis=0)         # self-norm
        acc_ht += np.einsum("bm,bmd->bd", (nk / n) / p, vk).sum(axis=0)    # HT, fixed p
        done += b
    return acc_sn / N_DRAWS, acc_ht / N_DRAWS


def main():
    print("Unbiased aggregation under client imbalance (FedAvg, 05-improvements.tex M.2)")
    print(f"K={K} clients, m=C*K={M}, dim={D}, {N_DRAWS:,} Monte-Carlo draws")
    print("Target vbar = sum_k (n_k/n) v_k; we report bias ||E[agg]-vbar||\n")

    sz_i, v_i = make_population(imbalanced=True)
    vbar_i = true_target(sz_i, v_i)
    sn_i, ht_i = mc_uniform(sz_i, v_i)
    bias_sn = float(np.linalg.norm(sn_i - vbar_i))
    bias_ht = float(np.linalg.norm(ht_i - vbar_i))

    sz_b, v_b = make_population(imbalanced=False)
    floor = float(np.linalg.norm(mc_uniform(sz_b, v_b)[0] - true_target(sz_b, v_b)))

    print(f"IMBALANCED  self-norm n_k/m_t (FedAvg)   bias = {bias_sn:.3e}   BIASED")
    print(f"IMBALANCED  Horvitz-Thompson (K/m)(n_k/n) bias = {bias_ht:.3e}   ~0 (unbiased)")
    print(f"BALANCED    self-norm n_k/m_t (sanity)    bias = {floor:.3e}   ~0 (MC floor)")
    print(f"\nSelf-normalized bias is {bias_sn / max(bias_ht, 1e-300):.0f}x the HT bias.")

    assert bias_sn > 10 * floor, "self-norm should be clearly biased under imbalance"
    assert bias_ht <= 5 * floor, "HT should sit at the MC floor (unbiased)"
    print("PASS -- Horvitz-Thompson removes the O(1/m) self-normalized bias.")


if __name__ == "__main__":
    main()

# --- run the witness ---
main()


# Adaptive Local-Work Schedule
**Level:** `extension`  *(novice | intermediate | advanced | graduate | frontier | paper | extension)*
**Concept ID:** `03-adaptive-local-work-schedule`
**Graph:** `improvements`
**Prerequisites:** [paper:07-local-update-count](../../paper/concepts/07-local-update-count.md), [paper:08-heterogeneity-gap-covariance](../../paper/concepts/08-heterogeneity-gap-covariance.md)
**Used by:** downstream nodes

## Plain-English intro
FedAvg fixes the local epochs $E$ for the whole run. But a large $E$ lets each client over-optimize its own (non-IID) data and drift out of the shared basin, so progress plateaus. The fix: treat $E$ like a learning rate — start large for fast early progress, then step-decay it toward FedSGD-like single steps ($E\to1$) so late-round clients stop overshooting.

## Symbols you'll see
| Symbol | Read aloud as | Meaning |
|--------|---------------|---------|
| $E_t$ | "E-sub-t" | Local epochs used in round $t$ (now scheduled, not fixed) <!-- TODO add to foundations --> |
| $E_{\min},E_{\max}$ | "E-min, E-max" | Floor / starting value of the schedule <!-- TODO add to foundations --> |
| $\tau$ | "tau" | Halving period: $E_t$ halves every $\tau$ rounds <!-- TODO add to foundations --> |
| $t$ | "t" | Communication-round index <!-- TODO add to foundations --> |
| $u_k=En_k/B$ | "u-sub-k" | Local SGD updates client $k$ runs per round <!-- TODO add to foundations --> |
| $E$ | "E" | Local epochs per client per round <!-- TODO add to foundations --> |
| $B$ | "B" | Local minibatch size ($B=\infty$ ⇒ full batch) <!-- TODO add to foundations --> |

## Formal definition
$$ E_t \;=\; \max\!\Big(E_{\min},\ \big\lfloor E_{\max}\,2^{-\lfloor t/\tau\rfloor}\big\rfloor\Big),\qquad t=0,1,\dots,R-1. $$
Large $E$ early; geometric step-decay toward FedSGD-like steps ($E_t\to E_{\min}=1$) late. The per-round local work $u_k=E_t n_k/B$ inherits the same decay.

## Why this matters
Dodges the large-$E$ plateau that paper node 08's heterogeneity drift $\Delta=\eta^2\mathrm{Cov}_k(H_k,g_k)$ predicts (02-math-deep-dive.md §3, Eq. 3.5) — the failure McMahan et al. flag (Fig. 3) but never fix. Implements 05-improvements.tex E.1 with no extra client state or communication.

## Code
The aligned runnable demo lives at [`../code/03-adaptive-local-work-schedule.py`](../code/03-adaptive-local-work-schedule.py). It demonstrates the concept on a finite/low-dimensional example, runnable in <30s on CPU.

**Expected output preview:**
```
E_t schedule  E_t = max(1, floor(64 * 2^-floor(t/12))):
  t=0:E=64 t=12:E=32 t=24:E=16 t=36:E=8 t=48:E=4
Final test accuracy over R=60 rounds, extreme non-IID FedAvg:
```

## Try it yourself
- Exercise 1: Set `SEP` larger (well-separated blobs). The drift vanishes and fixed-large $E$ stops plateauing — confirming the plateau is a heterogeneity effect, not a generic one.
- Exercise 2: Replace the $2^{-\lfloor t/\tau\rfloor}$ decay with a linear or cosine schedule on $E_t$; compare final accuracy against the geometric one.

## Further reading
- McMahan et al. 2017, *Communication-Efficient Learning of Deep Networks from Decentralized Data* (arXiv:1602.05629), §3 and Fig. 3.


In [ ]:
"""Adaptive Local-Work Schedule -- concept 03-adaptive-local-work-schedule of the
improvements learning map. Schedule the per-round local epochs E_t like a learning
rate (large early, decay toward FedSGD late) to dodge the large-E client-drift
plateau the FedAvg paper flags but never fixes (05-improvements.tex E.1).
Runnable code analog of concepts/03-adaptive-local-work-schedule.md.
"""
import numpy as np

np.random.seed(0)
np.seterr(over="ignore", divide="ignore", invalid="ignore")

D, C_CLS, K, C, LR, SEP = 12, 6, 40, 0.2, 4.0, 0.25  # small toy; SEP<1 => overlap
R, E_MIN, E_MAX, TAU = 60, 1, 64, 12                  # round budget + schedule knobs


def schedule(t):
    """E_t = max(E_min, floor(E_max * 2^(-floor(t/tau))))  (05-improvements.tex E.1)."""
    return max(E_MIN, int(E_MAX * 2.0 ** (-(t // TAU))))


def make_data(n, rng, centers):
    y = rng.integers(0, C_CLS, size=n)
    X = centers[y] + rng.normal(0, 1.0, size=(n, D)) * np.geomspace(1, 0.05, D)
    return X, y


def softmax(z):
    z = np.clip(z - z.max(1, keepdims=True), -60, 60)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)


def grad(W, X, y):
    P = softmax(X @ W)
    Y = np.zeros_like(P)
    Y[np.arange(len(y)), y] = 1.0
    return X.T @ (P - Y) / len(y)


def acc(W, X, y):
    return float((np.argmax(X @ W, 1) == y).mean())


def client_step(W, X, y, E):  # full-batch local SGD, E epochs from shared W
    W = W.copy()
    for _ in range(E):
        W -= LR * grad(W, X, y)
    return W


def fed_run(E_of_t, parts, Xtr, ytr, Xte, yte):
    rng = np.random.default_rng(1)
    W = np.zeros((D, C_CLS))                       # shared init each run
    m = max(1, int(round(C * K)))
    sizes = np.array([len(p) for p in parts])
    for t in range(R):
        S = rng.choice(K, m, replace=False)
        Wn = np.zeros_like(W)
        for k in S:                                # corrected n_k/m_t average
            Wk = client_step(W, Xtr[parts[k]], ytr[parts[k]], E_of_t(t))
            Wn += sizes[k] / sizes[S].sum() * Wk
        W = Wn
    return acc(W, Xte, yte)


def main():
    rng = np.random.default_rng(0)
    centers = rng.normal(0, SEP, (C_CLS, D)) * np.geomspace(1, 0.05, D)
    Xtr, ytr = make_data(2400, rng, centers)
    Xte, yte = make_data(1200, rng, centers)
    order = np.argsort(ytr, kind="stable")         # extreme non-IID: 1 class/client
    shards = np.array_split(order, K)
    parts = [shards[i] for i in rng.permutation(K)]

    print("E_t schedule  E_t = max(%d, floor(%d * 2^-floor(t/%d))):" % (E_MIN, E_MAX, TAU))
    print("  " + " ".join("t=%d:E=%d" % (t, schedule(t)) for t in range(0, R, TAU)))
    decayed = fed_run(schedule, parts, Xtr, ytr, Xte, yte)
    print("Final test accuracy over R=%d rounds, extreme non-IID FedAvg:" % R)
    best_fixed, best_E = -1.0, None
    for E in (1, 4, 16, 64):
        a = fed_run(lambda t, E=E: E, parts, Xtr, ytr, Xte, yte)
        tag = "  (fixed large-E plateau)" if E == E_MAX else ""
        print("  fixed E=%-3d  acc=%.2f%%%s" % (E, 100 * a, tag))
        if a > best_fixed:
            best_fixed, best_E = a, E
    print("  DECAYED %d->%d acc=%.2f%%  (best fixed=E=%d:%.2f%%)"
          % (E_MAX, E_MIN, 100 * decayed, best_E, 100 * best_fixed))
    print("VERDICT: decayed schedule beats best fixed E: %s (+%.2f pts)"
          % (decayed >= best_fixed - 1e-9, 100 * (decayed - best_fixed)))


if __name__ == "__main__":
    main()


# --- run the witness ---
main()


# Compute-Accounting Pareto
**Level:** `extension`  *(novice | intermediate | advanced | graduate | frontier | paper | extension)*
**Concept ID:** `04-compute-accounting-pareto`
**Graph:** `improvements`
**Prerequisites:** [paper:07-local-update-count](../../paper/concepts/07-local-update-count.md)
**Used by:** downstream nodes

## Plain-English intro
FedAvg reports only *rounds of communication*, justified by the unmeasured premise that on-device computation is "essentially free." But each round each client runs $u_k=En_k/B$ local SGD updates, and total on-device work scales as $R\cdot C\cdot K\cdot u_k$. If you plot rounds **and** total compute together, high-$u$ configs that win on the rounds axis can be *Pareto-dominated* on the compute axis. This node turns rounds-alone reporting into a rounds-vs-compute trade.

## Symbols you'll see
| Symbol | Read aloud as | Meaning |
|--------|---------------|---------|
| $u_k=En_k/B$ | "u-sub-k" | Local SGD updates client $k$ runs per round <!-- TODO add to foundations --> |
| $E$ | "E" | Local epochs per client per round <!-- TODO add to foundations --> |
| $n_k$ | "n-sub-k" | Number of examples on client $k$ <!-- TODO add to foundations --> |
| $B$ | "B" | Local minibatch size ($B=\infty\Rightarrow$ full batch, $u_k=1$) <!-- TODO add to foundations --> |
| $R$ | "R" | Rounds of communication to hit the target <!-- TODO add to foundations --> |
| $C$ | "C" | Fraction of clients selected per round <!-- TODO add to foundations --> |
| $K$ | "K" | Total number of clients <!-- TODO add to foundations --> |

## Formal definition
$$
u_k=\frac{E\,n_k}{B},\qquad
\text{Compute}_{\text{total}}\ \propto\ R\cdot C\cdot K\cdot u_k
\;=\; R\cdot(CK)\cdot\frac{E\,n_k}{B}.
$$
Report the **Pareto frontier** $\{(R,\ \text{Compute}_{\text{total}})\}$ over $(E,B)$ configs, not $R$ alone: a config is dominated if some other config has both fewer rounds and less total compute.

## Why this matters
Tests FedAvg's unquantified "computation is free" premise (Gaps Flagged, 02-math-deep-dive.md; 05-improvements.tex E.2, design): high-$u$ configurations such as $E{=}20,B{=}10$ ($u_k$ up to 1200/round) look best on rounds but may be Pareto-dominated on total compute, revealing a genuine knee.

## Code
The aligned runnable demo lives at [`../code/04-compute-accounting-pareto.py`](../code/04-compute-accounting-pareto.py). It demonstrates the concept on a finite/low-dimensional example, runnable in <30s on CPU.
**Expected output preview:**
```
Compute-Accounting Pareto for FedAvg (toy rounds-to-target, n_k=600, C*K=10)
config            u_k |  rounds R | total compute (R*CK*u_k) | Pareto?
E=1,  B=inf         1 |       40  |               4.00e+02   | on-frontier
```

## Try it yourself
- Exercise 1: Change the hardcoded rounds so a high-$u$ config also wins on compute; watch the frontier shrink to one point.
- Exercise 2: Add a per-round communication cost (one model up+down) and plot a 3-way rounds/compute/comm trade.

## Further reading
- McMahan et al. 2017, "Communication-Efficient Learning of Deep Networks from Decentralized Data" (arXiv:1602.05629), Table 2 and the "computation is free" remark.


In [ ]:
"""Compute-Accounting Pareto -- concept 04-compute-accounting-pareto of the improvements learning map.

FedAvg reports rounds-to-target alone, assuming on-device compute is "free". Here we
pair each (E,B) config's rounds R with its total local-update FLOP-proxy R*(C*K)*u_k,
revealing the rounds-vs-compute knee where a high-u config is Pareto-dominated.
Runnable code analog of concepts/04-compute-accounting-pareto.py.
"""
import numpy as np

# Toy fixed setup (matches the paper's MNIST family): n_k=600 examples/client,
# C*K = 10 clients selected per round. u_k = E*n_k/B, with B=inf => u_k=1.
N_K = 600          # examples per client
CK = 10            # C*K selected clients per round
INF = float("inf")


def updates_per_round(E, B):
    """u_k = E*n_k/B local SGD updates; B=inf means one full-batch step => u_k=1."""
    if B == INF:
        return float(E)            # E epochs, 1 batch each => u_k = E (and E=1 => FedSGD)
    return E * N_K / B


def main():
    np.random.seed(0)
    # Hardcoded VERIFIED vanilla-FedAvg rounds-to-target for a few (E,B) configs
    # (toy numbers in the spirit of Table 2: more local work -> fewer rounds).
    configs = [
        # (label,           E,   B,    rounds R)
        ("E=1,  B=inf",      1, INF,   40),   # FedSGD endpoint: cheap/round, many rounds
        ("E=5,  B=inf",      5, INF,   23),
        ("E=20, B=inf",     20, INF,   13),
        ("E=5,  B=10",       5,  10,   11),   # u_k=300: cheaper compute AND fewer rounds
        ("E=20, B=10",      20,  10,   12),   # u_k=1200: diminishing returns -> DOMINATED
    ]

    rows = []
    for label, E, B, R in configs:
        u = updates_per_round(E, B)
        compute = R * CK * u           # total on-device local-update FLOP-proxy
        rows.append((label, u, R, compute))

    # Pareto test on (rounds, compute): minimize BOTH. A config is dominated if some
    # other config has rounds <= and compute <= with at least one strictly less.
    def dominated(i):
        Ri, Ci = rows[i][2], rows[i][3]
        for j, (_, _, Rj, Cj) in enumerate(rows):
            if j == i:
                continue
            if Rj <= Ri and Cj <= Ci and (Rj < Ri or Cj < Ci):
                return True
        return False

    print("Compute-Accounting Pareto for FedAvg (toy rounds-to-target, n_k=600, C*K=10)")
    print("config            u_k |  rounds R | total compute (R*CK*u_k) | Pareto?")
    for i, (label, u, R, compute) in enumerate(rows):
        tag = "DOMINATED  " if dominated(i) else "on-frontier"
        print("{:<16}{:>5.0f} | {:>8d}  | {:>22.2e}   | {}".format(label, u, R, compute, tag))

    # Identify the knee: the on-frontier config minimizing rounds, and the one
    # minimizing compute -- the trade lives between them.
    frontier = [r for i, r in enumerate(rows) if not dominated(i)]
    fewest_rounds = min(frontier, key=lambda r: r[2])
    least_compute = min(frontier, key=lambda r: r[3])
    print("---")
    print("Fewest-rounds frontier config : {:<12} ({} rounds, compute {:.2e})".format(
        fewest_rounds[0], fewest_rounds[2], fewest_rounds[3]))
    print("Least-compute frontier config : {:<12} ({} rounds, compute {:.2e})".format(
        least_compute[0], least_compute[2], least_compute[3]))
    dom = [r[0] for i, r in enumerate(rows) if dominated(i)]
    print("Pareto-DOMINATED configs (lose on BOTH axes vs another): {}".format(
        dom if dom else "none"))
    print("LESSON: reporting rounds alone hides that the rounds-winner is not the "
          "compute-winner -- the knee is the trade FedAvg never measured (E.2).")


if __name__ == "__main__":
    main()


# --- run the witness ---
main()


# Permutation-Aligned Averaging
**Level:** `extension`
**Concept ID:** `05-permutation-aligned-averaging`
**Graph:** `improvements`
**Prerequisites:** [paper:12-shared-init-permutation-barrier](../../paper/concepts/12-shared-init-permutation-barrier.md), [linear assignment problem](https://github.com/pleyva2004/math-foundations/blob/main/concepts/linear-assignment-problem.md)
**Used by:** downstream nodes

## Plain-English intro
A one-hidden-layer net is unchanged if you relabel its hidden units, so two nets trained from independent inits land in *different* relabelings of the same solution and naive averaging blends unrelated units (the loss barrier of paper node 12). Before averaging, we instead solve a linear assignment problem for the permutation $P$ that best matches one net's hidden units to the other's, then average the aligned weights. This is Git Re-Basin weight-matching, and it relaxes FedAvg's shared-initialization requirement.

## Symbols you'll see
| Symbol | Read aloud as | Meaning |
|--------|---------------|---------|
| $W_1$ | "W-one" | Server model's first-layer (hidden) weight matrix <!-- TODO add to foundations --> |
| $W_1'$ | "W-one-prime" | Client model's first-layer weight matrix <!-- TODO add to foundations --> |
| $P$ | "P" | A permutation (matrix) relabeling the $H$ hidden units <!-- TODO add to foundations --> |
| $\Pi_H$ | "cap-Pi-sub-H" | Set of all $H\times H$ permutation matrices <!-- TODO add to foundations --> |
| $H$ | "H" | Number of hidden units <!-- TODO add to foundations --> |
| $\lVert\cdot\rVert_F$ | "Frobenius norm" | Entrywise Euclidean norm of a matrix <!-- TODO add to foundations --> |
| $w,\,w'$ | "w, w-prime" | Full parameter vectors of the server / client models <!-- TODO add to foundations --> |

## Formal definition
$$
P^\star=\arg\min_{P\in\Pi_H}\big\lVert W_1-W_1'P^\top\big\rVert_F^2
       =\arg\max_{P\in\Pi_H}\sum_{i=1}^{H}\big\langle (W_1)_{:,i},\,(W_1')_{:,P(i)}\big\rangle,
$$
since the column norms are permutation-invariant, only the cross-term depends on $P$ — a **linear assignment problem**. The aligned client $w'\mapsto P^\star(w')$ is a function-preserving symmetry ($W_1'\mapsto W_1'P^\top,\ b_1'\mapsto Pb_1',\ W_2'\mapsto PW_2'$), and the server averages $\tfrac12 w+\tfrac12 P^\star(w')$.

## Why this matters
Relaxes FedAvg's shared-init requirement (paper node 12) by aligning independently-trained client models into a common basin via Git Re-Basin weight-matching; this is proposal T.1 of `05-improvements.tex`.

## Code
The aligned runnable demo lives at [`../code/05-permutation-aligned-averaging.py`](../code/05-permutation-aligned-averaging.py). It demonstrates the concept on a finite/low-dimensional example, runnable in <30s on CPU.

**Expected output preview:**
```
Permutation-aligned averaging witness (Git Re-Basin weight-matching)
H = 6 hidden units; client is a relabeling Q = [0, 5, 3, 2, 1, 4]
Hidden-layer Frobenius distance ||W1 - W1'||_F  unmatched = 7.2038
```

## Try it yourself
- Exercise 1: Drop the noise (`0.05 -> 0.0`). The matched distance and `max |f_client - f_aligned|` both go to ~0, and the recovered permutation is exactly $Q^{-1}$.
- Exercise 2: Increase the noise (`0.05 -> 0.5`) until greedy matching mis-pairs a column; check that the matched distance still beats unmatched but the function is no longer preserved.

## Further reading
- Ainsworth, Hayase, Srinivasa, "Git Re-Basin: Merging Models modulo Permutation Symmetries," ICLR 2023.


In [ ]:
"""Permutation-Aligned Averaging -- concept 05-permutation-aligned-averaging of the improvements learning map.

Two one-hidden-layer MLPs from independent inits sit in different elements of the
hidden-unit permutation orbit; weight-matching recovers the permutation P that aligns
their hidden units so averaging blends matched (not unrelated) units.
Runnable code analog of concepts/05-permutation-aligned-averaging.py.md.
"""
import numpy as np


def frob(A, B):
    """Frobenius distance ||A - B||_F."""
    return float(np.linalg.norm(A - B))


def greedy_match(W1, W1p):
    """Greedy column assignment maximizing sum_i <W1[:,i], W1p[:,P(i)]>.

    Returns perm so that W1p[:, perm] best aligns column-by-column with W1.
    Equivalent target to argmin_P ||W1 - W1p P^T||_F^2 (cross-term is the only
    P-dependent part since the column norms are permutation-invariant).
    """
    H = W1.shape[1]
    score = W1.T @ W1p                       # score[i, j] = <W1[:,i], W1p[:,j]>
    perm = -np.ones(H, dtype=int)
    used = np.zeros(H, dtype=bool)
    order = np.dstack(np.unravel_index(np.argsort(-score, axis=None), score.shape))[0]
    for i, j in order:                       # descending inner products
        if perm[i] == -1 and not used[j]:
            perm[i] = j
            used[j] = True
    return perm


def main():
    np.random.seed(0)
    D, H = 4, 6                              # input dim, hidden units

    # Server model w = (W1, W2); independent-init client model w' = (W1p, W2p).
    W1 = np.random.randn(D, H)
    W2 = np.random.randn(H, 1)
    # Build the client as a TRUE permutation+noise of the server: a relabeling Q
    # of hidden units plus small training noise -> same orbit, different element.
    Q = np.random.permutation(H)
    W1p = W1[:, Q] + 0.05 * np.random.randn(D, H)
    W2p = W2[Q, :] + 0.05 * np.random.randn(H, 1)

    print("Permutation-aligned averaging witness (Git Re-Basin weight-matching)")
    print("H =", H, "hidden units; client is a relabeling Q =", Q.tolist())

    # (1) Recover the alignment permutation P from W1 vs W1'.
    perm = greedy_match(W1, W1p)             # perm[i] = which client col matches server col i
    inv = np.argsort(perm)                   # apply P consistently to client weights

    d_unmatched = frob(W1, W1p)
    d_matched = frob(W1, W1p[:, perm])
    print("Hidden-layer Frobenius distance ||W1 - W1'||_F  unmatched = %.4f" % d_unmatched)
    print("Hidden-layer Frobenius distance ||W1 - P(W1')||_F matched  = %.4f" % d_matched)
    print("matched < unmatched :", d_matched < d_unmatched,
          "(recovered perm == Q^-1 :", bool(np.array_equal(perm, np.argsort(Q))), ")")

    # (2) Function-preserving: applying P consistently (W1'->cols, W2'->rows) leaves
    # the client's function unchanged.  Compare outputs on random inputs.
    X = np.random.randn(20, D)
    relu = lambda Z: np.maximum(Z, 0.0)
    f_client = relu(X @ W1p) @ W2p
    W1p_al = W1p[:, perm]                    # permute hidden columns of layer 1
    W2p_al = W2p[perm, :]                    # permute matching rows of layer 2
    f_aligned = relu(X @ W1p_al) @ W2p_al
    max_fn_err = float(np.max(np.abs(f_client - f_aligned)))
    print("max |f_client - f_aligned| = %.2e  (permutation preserves the function)" % max_fn_err)

    # (3) Averaging: naive vs aligned, distance of the average to the server model.
    avg_naive = 0.5 * W1 + 0.5 * W1p
    avg_align = 0.5 * W1 + 0.5 * W1p_al
    print("dist(server, naive avg)   = %.4f" % frob(W1, avg_naive))
    print("dist(server, aligned avg) = %.4f" % frob(W1, avg_align))
    print("VERDICT:", "PASS" if frob(W1, avg_align) < frob(W1, avg_naive) and max_fn_err < 1e-9
          else "FAIL", "-- alignment yields a closer (compatible) average.")


if __name__ == "__main__":
    main()


# --- run the witness ---
main()


# Federated LoRA Communication
**Level:** `extension`  *(novice | intermediate | advanced | graduate | frontier | paper | extension)*
**Concept ID:** `06-federated-lora-communication`
**Graph:** `improvements`
**Prerequisites:** [paper:04-fedsgd-gradient-descent](../../paper/concepts/04-fedsgd-gradient-descent.md), [low-rank factorization](https://github.com/pleyva2004/math-foundations/blob/main/concepts/low-rank-factorization.md)
**Used by:** downstream nodes

## Plain-English intro
FedAvg's whole premise is that sending the model each round costs more than the on-device compute. For a big model that hurts: a full layer $W$ is $d\times d$ parameters. LoRA freezes $W$ (shared once) and trains only a thin low-rank adapter $\Delta W = BA$; running FedAvg over *just the adapter* drops per-round communication from $\Theta(d^2)$ to $\Theta(2rd)$, a fraction of $2r/d$.

## Symbols you'll see
| Symbol | Read aloud as | Meaning |
|--------|---------------|---------|
| $W\in\mathbb{R}^{d\times d}$ | "cap-W in R d-by-d" | Frozen base weight matrix (never communicated) <!-- TODO add to foundations --> |
| $\Delta W$ | "delta-W" | Trained low-rank update added to $W$ <!-- TODO add to foundations --> |
| $A\in\mathbb{R}^{r\times d}$ | "cap-A" | Down-projection factor of the adapter <!-- TODO add to foundations --> |
| $B\in\mathbb{R}^{d\times r}$ | "cap-B" | Up-projection factor of the adapter <!-- TODO add to foundations --> |
| $r$ | "r" | Adapter rank, $r\ll d$ <!-- TODO add to foundations --> |
| $d$ | "d" | Layer width / parameter dimension <!-- TODO add to foundations --> |
| $w$ | "w" | The FedAvg model — here only the adapter $(A,B)$ <!-- TODO add to foundations --> |

## Formal definition
$$
W\in\mathbb{R}^{d\times d}\ \text{frozen},\qquad
\Delta W = BA,\quad B\in\mathbb{R}^{d\times r},\ A\in\mathbb{R}^{r\times d},\ \operatorname{rank}(\Delta W)\le r,
$$
$$
\text{FedAvg over } w=(A,B):\qquad
\underbrace{\lvert A\rvert+\lvert B\rvert = 2rd}_{\text{per-round comms}}\ \;\text{vs.}\;\ \underbrace{\lvert W\rvert = d^2}_{\text{full model}},
\qquad \text{ratio}=\frac{2rd}{d^2}=\frac{2r}{d}.
$$

## Why this matters
Makes FedAvg's compute-for-communication trade dramatic for large models: per-round comms become $\Theta(2rd)\ll\Theta(d^2)$, the modern realization of the communication-reduction program of konecny2016 (05-improvements.tex T.2, design).

## Code
The aligned runnable demo lives at [`../code/06-federated-lora-communication.py`](../code/06-federated-lora-communication.py). It demonstrates the concept on a finite/low-dimensional example, runnable in <30s on CPU.

**Expected output preview:**
```
Federated LoRA communication (05-improvements.tex T.2)
Freeze base W in R^{d x d}; FedAvg only adapter Delta W = B A,
  A in R^{r x d}, B in R^{d x r}  ->  per-round comms Theta(2rd) vs Theta(d^2).
```

## Try it yourself
- Exercise 1: Add a non-square base $W\in\mathbb{R}^{d_\text{out}\times d_\text{in}}$ and rederive the ratio (now $r(d_\text{in}+d_\text{out})/(d_\text{in}d_\text{out})$).
- Exercise 2: Pick a target budget (say 1% comms) and solve $2r/d$ for the largest admissible rank $r$ at each $d$ in the table.

## Further reading
- 05-improvements.tex, section T.2; sandbox `real_fedlora.py` (FedAvg over LoRA adapters, ~4.3% of params).
- Konecny et al., *Federated Learning: Strategies for Improving Communication Efficiency*, arXiv:1610.05492.


In [ ]:
"""Federated LoRA Communication -- concept 06-federated-lora-communication
of the improvements learning map.

Freezing a base weight W (d x d) and FedAvg-ing only a low-rank adapter
Delta W = B A (B: d x r, A: r x d) cuts per-round communication from Theta(d^2)
to Theta(2 r d); the ratio 2r/d shrinks as the base model grows. This witness
tabulates base vs adapter param counts and the communication ratio for several
(d, r), including a ~4.3% case matching the sandbox real_fedlora.py.

Runnable code analog of concepts/06-federated-lora-communication.md.
"""

import numpy as np

np.random.seed(0)


def counts(d, r):
    """Base params (d^2), adapter params (2*r*d), and comm ratio (2r/d)."""
    base = d * d
    adapter = 2 * r * d
    ratio = adapter / base          # == 2r/d, the per-round comm fraction
    return base, adapter, ratio


def main():
    print("Federated LoRA communication (05-improvements.tex T.2)")
    print("Freeze base W in R^{d x d}; FedAvg only adapter Delta W = B A,")
    print("  A in R^{r x d}, B in R^{d x r}  ->  per-round comms Theta(2rd) vs Theta(d^2).\n")

    # (d, r) pairs spanning toy -> large; the last targets ~4.3% (sandbox figure).
    configs = [(64, 8), (256, 8), (1024, 16), (4096, 16), (1024, 22)]

    header = "{:>7} {:>5} | {:>14} {:>14} | {:>10} {:>9}".format(
        "d", "r", "base d^2", "adapter 2rd", "ratio 2r/d", "savings"
    )
    print(header)
    print("-" * len(header))
    for d, r in configs:
        base, adapter, ratio = counts(d, r)
        # cross-check the closed form 2r/d against the explicit param counts
        assert abs(ratio - (2 * r / d)) < 1e-12
        assert adapter < base, "LoRA only saves when 2rd < d^2, i.e. 2r < d"
        print("{:>7} {:>5} | {:>14,} {:>14,} | {:>9.2f}% {:>8.1f}x".format(
            d, r, base, adapter, 100.0 * ratio, base / adapter
        ))

    # Numerical witness that B A truly reconstructs a rank-r matrix of shape d x d
    # (so the adapter, not the base, carries the trained update).
    d, r = 1024, 22
    A = np.random.normal(0.0, 0.02, size=(r, d))
    B = np.random.normal(0.0, 0.02, size=(d, r))
    # errstate guards a benign numpy-2.0 + Accelerate BLAS SIMD quirk on Apple
    # Silicon that raises spurious FP flags here; the product is finite & valid.
    with np.errstate(divide="ignore", over="ignore", invalid="ignore"):
        dW = B @ A
    assert np.isfinite(dW).all()
    rank = int(np.linalg.matrix_rank(dW))
    base, adapter, ratio = counts(d, r)
    print("\nWitness (d={}, r={}): Delta W = B@A has shape {} and rank {} (<= r).".format(
        d, r, dW.shape, rank
    ))
    print("  base d^2 = {:,} params; adapter 2rd = {:,} params communicated/round.".format(
        base, adapter
    ))
    print("  per-round communication ratio 2r/d = {:.1f}%  ({:.0f}x less than the base).".format(
        100.0 * ratio, base / adapter
    ))
    assert rank <= r, "B@A cannot exceed rank r"
    print("PASS -- adapter is rank <= r and costs only 2r/d of the base each round.")


if __name__ == "__main__":
    main()


# --- run the witness ---
main()


# Effective-ODE Averaging Bound
**Level:** `extension`  *(novice | intermediate | advanced | graduate | frontier | paper | extension)*
**Concept ID:** `07-effective-ode-averaging-bound`
**Graph:** `improvements`
**Prerequisites:** [paper:08-heterogeneity-gap-covariance](../../paper/concepts/08-heterogeneity-gap-covariance.md), [paper:07-local-update-count](../../paper/concepts/07-local-update-count.md), [taylor theorem](https://github.com/pleyva2004/math-foundations/blob/main/concepts/taylor-theorem.md), [hessian](https://github.com/pleyva2004/math-foundations/blob/main/concepts/hessian.md), [covariance](https://github.com/pleyva2004/math-foundations/blob/main/concepts/covariance.md)
**Used by:** downstream nodes

## Plain-English intro
The math deep dive guessed FedAvg's $E>1$ drift from a two-step Taylor sketch: $\Delta=\eta^2\,\mathrm{Cov}_k(H_k,g_k)$. That was a *heuristic*. Here we make it a *theorem*. Idealize each client's $u_k$ local full-batch steps as the **gradient flow** of its loss $F_k$ run for continuous time $T=\eta u_k$. The FedAvg server map is the weighted average $\mathcal S^T=\sum_k\beta_k\Phi_k^T$ of those per-client flows; the centralized reference is the single flow $\Phi_f^T$ of the global $f$. A Lie–Taylor expansion shows they agree to second order, with the *exact* leading discrepancy $\tfrac{T^2}{2}\lVert\mathrm{Cov}_k(H_k,g_k)\rVert$. This corrects the heuristic prefactor — the frozen $\eta^2$ should be $T^2/2=(\eta u_k)^2/2$, so the heuristic underpredicts by $u_k^2/4$ (about $6\times$ at $E=5$) — and shows $(E,B,\eta)$ enter only through the single invariant $T=\eta E n_k/B$.

## Symbols you'll see
| Symbol | Read aloud as | Meaning |
|--------|---------------|---------|
| $\Phi_k^{T}(w)$ | "cap-Phi-sub-k of T" | Gradient flow of $F_k$, $\dot\phi=-\nabla F_k(\phi)$, run for time $T$ from $w$ <!-- TODO add to foundations --> |
| $\Phi_f^{T}(w)$ | "cap-Phi-sub-f of T" | Gradient flow of the global $f=\sum_k\beta_kF_k$ for time $T$ <!-- TODO add to foundations --> |
| $\mathcal S^{T}(w)$ | "cap-S of T" | FedAvg server map $\sum_k\beta_k\Phi_k^{T}(w)$ (weighted average of flows) <!-- TODO add to foundations --> |
| $T=\eta u_k$ | "T" | Continuous horizon = (lr) $\times$ (local updates); $T=\eta E n_k/B$ <!-- TODO add to foundations --> |
| $H_k=\nabla^2 F_k(w)$ | "H-sub-k" | Client $k$'s local Hessian at the broadcast point $w$ <!-- TODO add to foundations --> |
| $g_k=\nabla F_k(w)$ | "g-sub-k" | Client $k$'s local gradient at $w$ <!-- TODO add to foundations --> |
| $\beta_k$ | "beta-sub-k" | Client weight $n_k/n$, $\sum_k\beta_k=1$ <!-- TODO add to foundations --> |
| $\Gamma=\mathrm{Cov}_k(H_k,g_k)$ | "cap-Gamma" | Curvature–gradient covariance across clients; $\sigma_{HG}=\lVert\Gamma\rVert$ <!-- TODO add to foundations --> |

## Formal definition
$$
\mathcal S^{T}(w)=\sum_{k=1}^K\beta_k\,\Phi_k^{T}(w),\qquad
\Phi_k^{T}(w)=w-T\,g_k+\tfrac{T^2}{2}\,H_kg_k+R_k(T),\quad \lVert R_k(T)\rVert\le\tfrac{T^3}{6}C_3 .
$$
For clients sharing a horizon $T$, averaging the flow expansions and subtracting $\Phi_f^{T}$ (the $O(1)$ term $w$ and the $O(T)$ term $-Tg$ cancel exactly) gives
$$
\boxed{\;\big\lVert \mathcal S^{T}(w)-\Phi_f^{T}(w)\big\rVert=\frac{T^2}{2}\,\sigma_{HG}+O(T^3),\qquad \sigma_{HG}=\big\lVert\mathrm{Cov}_k(H_k,g_k)\big\rVert\;}
$$
with explicit remainder $\lVert R(T)\rVert\le\tfrac{T^3}{3}C_3$ under bounded third derivatives, and leading drift direction exactly $\Gamma/\lVert\Gamma\rVert$. The single invariant is $T=\eta E n_k/B$.

## Why this matters
Turns the math deep dive's **heuristic** $\Delta=\eta^2\mathrm{Cov}_k(H_k,g_k)$ (02-math-deep-dive.md §3, Eq. 3.5) into a proved **theorem** and *corrects its prefactor*: the right magnitude is $T^2/2=(\eta u_k)^2/2$, not the frozen $\eta^2$ (the heuristic underpredicts by $u_k^2/4$, $\approx 6\times$ at $E=5$). Recovers the heuristic exactly as the $E{=}2,B{=}\infty$ ($T{=}2\eta$) case. Implements 05-improvements.tex T.3; its unequal-horizon remark $-\mathrm{Cov}_k(T_k,g_k)$ is the size-imbalance drift that node 08 (horizon-equalized local flow) cancels.

## Code
The aligned runnable demo lives at [`../code/07-effective-ode-averaging-bound.py`](../code/07-effective-ode-averaging-bound.py). It demonstrates the concept on a finite/low-dimensional example, runnable in <30s on CPU.

**Expected output preview:**
```
Effective-ODE averaging-drift bound (05-improvements.tex T.3)
Server map S^T = sum_k beta_k Phi_k^T  vs centralized flow Phi_f^T.
K=4 clients, dim=6, ||Cov_k(H_k,g_k)|| = sigma_HG = 0.0694

Measured drift scales as T^p with p ~ 2 (predicted exactly 2):
```

## Try it yourself
- Exercise 1: Set the two client Hessians equal (the homogeneous limit). Confirm $\Gamma=\mathrm{Cov}_k(H_k,g_k)=0$ and the drift collapses to the $O(T^3)$ remainder.
- Exercise 2: Hold $T$ fixed and grow $E$ (shrink the Euler step $\eta=T/u_k$). Confirm the discrete-FedAvg drift converges to the flow drift — the discretization-invariance of Lemma (discrete-to-flow consistency).

## Further reading
- 05-improvements.tex, section T.3; full proof in `proofs/effective-ode-averaging-bound.tex`.
- McMahan et al. 2017, *Communication-Efficient Learning of Deep Networks from Decentralized Data* (arXiv:1602.05629), §3 and Fig. 3.


In [ ]:
"""Effective-ODE Averaging Bound -- concept 07-effective-ode-averaging-bound of the
improvements learning map.

Idealize each client's u_k local full-batch steps as the gradient flow Phi_k^T of
its loss F_k run for time T = eta*u_k. The FedAvg server map S^T = sum_k beta_k
Phi_k^T is compared to the centralized flow Phi_f^T of the global f = sum_k beta_k
F_k. A Lie-Taylor expansion gives the EXACT leading drift

    || S^T(w) - Phi_f^T(w) || = (T^2/2) * ||Cov_k(H_k, g_k)|| + O(T^3),

with leading direction Cov_k(H_k,g_k)/||.||. This witness (i) measures the drift's
log-log slope vs T (predicted 2), (ii) checks D_t / pred -> ~1 as T -> 0, (iii)
checks the drift direction matches Cov_k(H_k,g_k) to cosine ~1, and (iv) shows the
heuristic frozen-eta^2 prefactor underpredicts by u_k^2/4 (~6x at E=5).
Runnable code analog of concepts/07-effective-ode-averaging-bound.md.
"""
import numpy as np

np.random.seed(0)
np.seterr(over="ignore", divide="ignore", invalid="ignore")


def make_clients(K=4, d=6):
    """K heterogeneous local quadratics F_k(w) = 0.5 (w-b_k)^T A_k (w-b_k).
    grad F_k(w) = A_k (w-b_k);  Hessian H_k = A_k (constant). Heterogeneous
    A_k => Cov_k(H_k,g_k) != 0, the curvature-gradient covariance that drifts.
    Hessians are kept SMALL and well-conditioned so the O(T^3) remainder
    (~ T^3 * (L^2 G)/3) is negligible against the leading (T^2/2)*sigma_HG term
    over the measured horizons -- i.e. we stay inside the bound's small-T regime."""
    beta = np.full(K, 1.0 / K)               # equal client weights n_k/n
    A, b = [], []
    for _ in range(K):
        M = 0.15 * np.random.randn(d, d)
        A.append(M @ M.T + 0.5 * np.eye(d))  # SPD, spectral norm O(1)
        b.append(0.5 * np.random.randn(d))
    return beta, A, b


def flow(A_k, b_k, w, T, steps):
    """Phi_k^T(w): gradient flow dw/dt = -A_k (w-b_k), integrated to time T by
    fine-grained explicit Euler (steps large => ~exact continuous flow)."""
    dt = T / steps
    for _ in range(steps):
        w = w - dt * (A_k @ (w - b_k))
    return w


def main():
    beta, A, b = make_clients()
    d = A[0].shape[0]
    w = np.random.randn(d)
    K = len(A)

    # Global objective f = sum_k beta_k F_k: Hessian H = sum beta_k A_k, and its
    # gradient flow uses grad f(w) = sum_k beta_k A_k (w - b_k).
    H = sum(beta[k] * A[k] for k in range(K))
    g = sum(beta[k] * (A[k] @ (w - b[k])) for k in range(K))

    def server_map(T, steps):
        return sum(beta[k] * flow(A[k], b[k], w.copy(), T, steps) for k in range(K))

    def central_flow(T, steps):
        wf = w.copy()
        dt = T / steps
        for _ in range(steps):
            wf = wf - dt * sum(beta[k] * (A[k] @ (wf - b[k])) for k in range(K))
        return wf

    # The theorem's leading object: Gamma = Cov_k(H_k, g_k) = sum beta_k H_k g_k - H g.
    gk = [A[k] @ (w - b[k]) for k in range(K)]
    Gamma = sum(beta[k] * (A[k] @ gk[k]) for k in range(K)) - H @ g
    sigma_HG = np.linalg.norm(Gamma)

    print("Effective-ODE averaging-drift bound (05-improvements.tex T.3)")
    print("Server map S^T = sum_k beta_k Phi_k^T  vs centralized flow Phi_f^T.")
    print("K=%d clients, dim=%d, ||Cov_k(H_k,g_k)|| = sigma_HG = %.4f\n" % (K, d, sigma_HG))

    # (1) measure drift at a geometric sweep of small horizons T; fit log-log slope.
    Ts = np.geomspace(2e-3, 4e-2, 7)
    steps = 6000                              # fine Euler => essentially the flow
    drifts, ratios = [], []
    for T in Ts:
        D = np.linalg.norm(server_map(T, steps) - central_flow(T, steps))
        pred = 0.5 * T ** 2 * sigma_HG
        drifts.append(D)
        ratios.append(D / pred)
    drifts = np.array(drifts)
    slope = np.polyfit(np.log(Ts), np.log(drifts), 1)[0]

    print("Measured drift scales as T^p with p ~ 2 (predicted exactly 2):")
    print("   T        drift D_t     pred=(T^2/2)sigma   D_t/pred")
    for T, D, r in zip(Ts, drifts, ratios):
        print("  %.4f   %.4e      %.4e        %.4f" % (T, D, 0.5 * T ** 2 * sigma_HG, r))
    print("  log-log slope of D_t vs T = %.3f  (predicted 2.0)" % slope)
    print("  D_t/pred -> %.3f as T -> 0  (predicted 1.0)" % ratios[0])

    # (2) direction: leading drift should point along Gamma/||Gamma||.
    Tsmall = Ts[0]
    drift_vec = server_map(Tsmall, steps) - central_flow(Tsmall, steps)
    cos = float(drift_vec @ Gamma / (np.linalg.norm(drift_vec) * sigma_HG))
    print("  drift direction . Cov_k(H_k,g_k) cosine = %.4f  (predicted ~1.0)\n" % cos)

    # (3) the heuristic Delta = eta^2 Cov_k(H_k,g_k) FREEZES the prefactor at eta^2;
    # (3) the heuristic Delta = eta^2 Cov_k(H_k,g_k) is the two-step Taylor sketch:
    # it equals the EXACT law only at its calibration point E=2, B=inf (T=2*eta),
    # where the correct prefactor is (2*eta)^2/2 = 2*eta^2. Read against that
    # reference, the correct prefactor T^2/2 at general u_k overshoots the frozen
    # heuristic by u_k^2/4 (~6x at E=5): the heuristic UNDERPREDICTS the drift.
    print("Heuristic prefactor vs theorem at fixed eta, varying E (B=inf => u_k=E):")
    eta = 0.05
    heuristic = 2.0 * eta ** 2                  # two-step Taylor sketch = exact at E=2
    for E in (2, 5, 10):
        u_k = E                               # B=inf, full batch => u_k = E
        T = eta * u_k
        correct = 0.5 * T ** 2                 # theorem prefactor (T^2/2)
        underpred = correct / heuristic        # = u_k^2/4
        tag = "  <- heuristic = exact (E=2 calibration point)" if E == 2 else ""
        print("  E=%-3d u_k=%-3d  T=%.3f | theorem T^2/2=%.4e  heuristic 2*eta^2=%.4e"
              "  underpred x%.2f (=u_k^2/4)%s"
              % (E, u_k, T, correct, heuristic, underpred, tag))

    ok_slope = abs(slope - 2.0) < 0.15
    ok_ratio = abs(ratios[0] - 1.0) < 0.15
    ok_dir = cos > 0.99
    ok_pref = abs((0.5 * (eta * 5) ** 2) / (2.0 * eta ** 2) - 5 ** 2 / 4.0) < 1e-9
    ok = ok_slope and ok_ratio and ok_dir and ok_pref
    print("\nVERDICT: %s -- drift = (T^2/2)||Cov_k(H_k,g_k)|| + O(T^3); slope=%.2f, "
          "ratio->%.2f, cos=%.4f, heuristic underpredicts by u_k^2/4."
          % ("PASS" if ok else "FAIL", slope, ratios[0], cos))
    assert ok, "effective-ODE drift law must hold to O(T^2)"



# --- run the witness ---
main()


# Horizon-Equalized Local Flow
**Level:** `extension`  *(novice | intermediate | advanced | graduate | frontier | paper | extension)*
**Concept ID:** `08-horizon-equalized-local-flow`
**Graph:** `improvements`
**Prerequisites:** [paper:07-local-update-count](../../paper/concepts/07-local-update-count.md), [paper:08-heterogeneity-gap-covariance](../../paper/concepts/08-heterogeneity-gap-covariance.md), [gradient descent](https://github.com/pleyva2004/math-foundations/blob/main/concepts/gradient-descent.md), [covariance](https://github.com/pleyva2004/math-foundations/blob/main/concepts/covariance.md)
**Used by:** downstream nodes

## Plain-English intro
One step of local SGD $w\leftarrow w-\eta\,\nabla F_k(w)$ is one forward-Euler step of the gradient flow $\dot w=-\nabla F_k(w)$, so a round of $u_k=En_k/B$ local steps integrates client $k$'s flow for physical time $T_k=\eta u_k=\eta E n_k/B$. Vanilla FedAvg fixes $(E,B,\eta)$ globally, so under client-size imbalance $T_k$ *varies*: the server averages models that each flowed a different amount of time toward their own local optimum. The effective-ODE bound (node 07, its unequal-horizon remark) shows this injects a **first-order** size-imbalance drift $-\mathrm{Cov}_k(T_k,g_k)$ — a pure *size-heterogeneity* term, present even if every $F_k$ has identical shape. The fix: give each client a per-round learning rate $\eta_k=T^\star/u_k$ so $T_k\equiv T^\star$ for all $k$, with $T^\star=\eta E\bar n/B$ (the horizon of an average-size client). Total flow-time is unchanged on average; only its distribution across clients is equalized. No extra communication, no client state.

## Symbols you'll see
| Symbol | Read aloud as | Meaning |
|--------|---------------|---------|
| $u_k=En_k/B$ | "u-sub-k" | Local SGD updates (forward-Euler steps) client $k$ runs per round <!-- TODO add to foundations --> |
| $T_k=\eta u_k$ | "T-sub-k" | Client $k$'s gradient-flow horizon, $=\eta E n_k/B$ <!-- TODO add to foundations --> |
| $T^\star=\eta E\bar n/B$ | "T-star" | Target horizon: that of an average-size client under the baseline LR <!-- TODO add to foundations --> |
| $\eta_k=T^\star/u_k$ | "eta-sub-k" | Per-client equalizing learning rate <!-- TODO add to foundations --> |
| $\bar n=n/K$ | "n-bar" | Mean examples per client <!-- TODO add to foundations --> |
| $g_k=\nabla F_k(w)$ | "g-sub-k" | Client $k$'s local gradient at the broadcast point <!-- TODO add to foundations --> |
| $\mathrm{Cov}_k(T_k,g_k)$ | "cov of T-k and g-k" | The size-imbalance drift cancelled when $T_k\equiv T^\star$ <!-- TODO add to foundations --> |

## Formal definition
$$
\eta_k=\frac{T^\star}{u_k}=\frac{T^\star B}{E\,n_k},\qquad
T^\star=\frac{\eta E\,\bar n}{B}\qquad\Longrightarrow\qquad
T_k=\eta_k u_k\equiv T^\star\ \ \forall k.
$$
Equalizing $T_k$ cancels the first-order size-imbalance drift $-\mathrm{Cov}_k(T_k,g_k)$ of node 07's unequal-horizon remark; the $T^\star$ choice keeps the average flow-time fixed (no free lunch), changing only its distribution across clients.

## Why this matters
Cancels the previously-unnamed size-imbalance drift $-\mathrm{Cov}_k(T_k,g_k)$ that vanilla FedAvg injects under client-size imbalance (the unequal-horizon remark of node 07's effective-ODE bound; 02-math-deep-dive.md §3). Implements 05-improvements.tex E.3 with no extra client state or communication — the actionable corollary of the T.3 theorem.

## Code
The aligned runnable demo lives at [`../code/08-horizon-equalized-local-flow.py`](../code/08-horizon-equalized-local-flow.py). It demonstrates the concept on a finite/low-dimensional example, runnable in <30s on CPU.

**Expected output preview:**
```
Horizon-equalized local flow (05-improvements.tex E.3)
eta_k = T*/u_k so T_k = eta_k*u_k = T* for every client (T* = eta*E*n_bar/B).
Rounds to target, pathological NON-IID, imbalanced (log-uniform) sizes:
  both arms share partition+sampling seed; differ ONLY in eta_k.
```

## Try it yourself
- Exercise 1: Set the size spread to 1 (balanced). Confirm $\eta_k\equiv\eta$, the two arms coincide exactly, and the result is a clean no-op (tie).
- Exercise 2: Grow the size spread and watch the baseline-minus-proposed round margin widen monotonically — the size-imbalance drift $-\mathrm{Cov}_k(T_k,g_k)$ grows with the spread.

## Further reading
- 05-improvements.tex, section E.3; full theorem in `proofs/effective-ode-averaging-bound.tex` (unequal-horizon remark).
- McMahan et al. 2017, *Communication-Efficient Learning of Deep Networks from Decentralized Data* (arXiv:1602.05629), §3.


In [ ]:
"""Horizon-Equalized Local Flow -- concept 08-horizon-equalized-local-flow of the
improvements learning map.

A round of local SGD is forward-Euler integration of dw/dt = -grad F_k(w) for
physical time T_k = eta*u_k = eta*E*n_k/B. Vanilla FedAvg fixes (E,B,eta) globally,
so under client-size imbalance T_k VARIES, injecting a first-order size-imbalance
drift -Cov_k(T_k,g_k) (the unequal-horizon remark of node 07's effective-ODE
bound). Fix: per-client eta_k = T*/u_k with T* = eta*E*n_bar/B, so T_k = T* for all
clients (total flow-time unchanged on average; only its distribution equalized).

This tiny FedAvg witness runs BASELINE (global eta) vs PROPOSED (eta_k = T*/u_k) on
a pathological non-IID partition with log-uniform client sizes; both arms share the
partition + sampling seed and differ ONLY in eta_k. It is a clean no-op when
balanced and needs fewer mean rounds as imbalance grows.
Runnable code analog of concepts/08-horizon-equalized-local-flow.md.
"""
import numpy as np

np.random.seed(0)
np.seterr(over="ignore", divide="ignore", invalid="ignore")

D, C_CLS, K, C, LR = 12, 6, 40, 0.2, 4.0      # small toy softmax-regression FedAvg
E, B, R, TARGET = 5, 10, 150, 0.90            # local epochs, batch, round cap, target acc
N_TRAIN, N_TEST = 2400, 1200
N_BAR = N_TRAIN / K                            # mean examples per client
REALIZATIONS = 6                               # average out integer-round noise
SEP = 2.5                                      # blob separation (headroom above target)


def make_data(n, rng, centers):
    y = rng.integers(0, C_CLS, size=n)
    X = centers[y] + rng.normal(0, 1.0, size=(n, D)) * np.geomspace(1, 0.05, D)
    return X, y


def softmax(z):
    z = np.clip(z - z.max(1, keepdims=True), -60, 60)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)


def grad(W, X, y):
    P = softmax(X @ W)
    Y = np.zeros_like(P)
    Y[np.arange(len(y)), y] = 1.0
    return X.T @ (P - Y) / len(y)


def acc(W, X, y):
    return float((np.argmax(X @ W, 1) == y).mean())


def u_of(nk):
    """u_k = E * ceil(n_k / B): forward-Euler steps client k runs per round."""
    return E * int(np.ceil(nk / min(B, nk)))


def client_step(W, X, y, lr):                  # E epochs of minibatch SGD, lr per step
    W = W.copy()
    nk = len(y)
    bsz = min(B, nk)
    rng = np.random.default_rng(0)
    for _ in range(E):
        perm = rng.permutation(nk)
        for s in range(0, nk, bsz):
            sel = perm[s:s + bsz]
            W -= lr * grad(W, X[sel], y[sel])
    return W


def partition(y, rng, spread):
    """Pathological non-IID (1-2 classes/client) with log-uniform sizes; mean ~N_BAR.
    spread = max/min size ratio; spread<=1 => balanced (all sizes == N_BAR)."""
    order = np.argsort(y, kind="stable")
    shards = np.array_split(order, K)
    pools = [shards[i] for i in rng.permutation(K)]
    if spread <= 1.0:
        sizes = np.full(K, int(round(N_BAR)), dtype=int)
    else:
        raw = np.exp(rng.uniform(0.0, np.log(spread), size=K))
        sizes = np.clip(np.round(raw / raw.mean() * N_BAR), 4, None).astype(int)
    parts = []
    for c in range(K):
        pool = pools[c]
        nk = int(sizes[c])
        sel = rng.choice(pool, size=nk, replace=(nk > len(pool)))
        parts.append(sel)
    return parts


def fed_run(parts, Xtr, ytr, Xte, yte, seed, equalize):
    rng = np.random.default_rng(seed)
    W = np.zeros((D, C_CLS))                    # SHARED init
    m = max(1, int(round(C * K)))
    sizes = np.array([len(p) for p in parts])
    T_star = LR * u_of(int(round(N_BAR)))       # horizon of an avg-size client
    for t in range(1, R + 1):
        S = rng.choice(K, m, replace=False)
        Wn = np.zeros_like(W)
        for k in S:
            lr_k = (T_star / u_of(sizes[k])) if equalize else LR
            Wk = client_step(W, Xtr[parts[k]], ytr[parts[k]], lr_k)
            Wn += sizes[k] / sizes[S].sum() * Wk    # corrected weighted average
        W = Wn
        if acc(W, Xte, yte) >= TARGET:
            return t
    return R + 1


def main():
    rng = np.random.default_rng(0)
    centers = rng.normal(0, SEP, (C_CLS, D)) * np.geomspace(1, 0.05, D)
    Xtr, ytr = make_data(N_TRAIN, rng, centers)
    Xte, yte = make_data(N_TEST, rng, centers)

    print("Horizon-equalized local flow (05-improvements.tex E.3)")
    print("eta_k = T*/u_k so T_k = eta_k*u_k = T* for every client (T* = eta*E*n_bar/B).")
    print("Rounds to target, pathological NON-IID, imbalanced (log-uniform) sizes:")
    print("  both arms share partition+sampling seed; differ ONLY in eta_k.\n")
    print("  spread~  n_min  n_max | BASE rnds  PROP rnds   mean dlt   W/L/T   faster?")
    print("  " + "-" * 70)

    deltas, spreads = [], []
    balanced_delta = None
    for spread in (1.0, 5.0, 20.0, 80.0):
        b_rounds, p_rounds = [], []
        nmins, nmaxs, real = [], [], []
        w = l = ti = 0
        for s in range(REALIZATIONS):
            prng = np.random.default_rng(1000 + s)
            parts = partition(ytr, prng, spread)
            sizes = np.array([len(p) for p in parts])
            nmins.append(int(sizes.min())); nmaxs.append(int(sizes.max()))
            real.append(sizes.max() / max(1, sizes.min()))
            rb = fed_run(parts, Xtr, ytr, Xte, yte, 2000 + s, equalize=False)
            rp = fed_run(parts, Xtr, ytr, Xte, yte, 2000 + s, equalize=True)
            b_rounds.append(rb); p_rounds.append(rp)
            w += rb > rp; l += rb < rp; ti += rb == rp
        bm, pm = float(np.mean(b_rounds)), float(np.mean(p_rounds))
        dlt = bm - pm
        rs = float(np.mean(real))
        if spread <= 1.0:
            balanced_delta = (dlt, l)
        else:
            deltas.append(dlt); spreads.append(rs)
        verdict = "PROPOSED" if dlt > 1e-9 else ("baseline" if dlt < -1e-9 else "tie")
        print("  %7.1f%6d%7d | %9.2f%10.2f%11.2f  %d/%d/%d %9s"
              % (rs, min(nmins), max(nmaxs), bm, pm, dlt, w, l, ti, verdict))
    print("  " + "-" * 70)

    # prediction checks (mirror the prototype's P1/P2/P3)
    p1 = abs(balanced_delta[0]) <= 0.5 and balanced_delta[1] == 0
    p2 = all(d > 1e-9 for d in deltas)
    p3 = all(deltas[i + 1] >= deltas[i] - 1e-9 for i in range(len(deltas) - 1))
    print("  (P1) balanced no-op (|dlt|<=0.5, 0 losses): dlt=%.2f -> %s"
          % (balanced_delta[0], "OK" if p1 else "FAIL"))
    print("  (P2) faster on every imbalanced spread: deltas=%s -> %s"
          % ([round(d, 2) for d in deltas], "OK" if p2 else "FAIL"))
    print("  (P3) margin widens with imbalance: %s -> %s"
          % ([round(d, 2) for d in deltas], "OK" if p3 else "FAIL"))
    ok = p1 and p2 and p3
    print("VERDICT: %s -- no-op when balanced; fewer mean rounds under imbalance, "
          "margin widens monotonically." % ("PASS" if ok else "MIXED/FAIL"))



# --- run the witness ---
main()
